# GeoCadastra: low-cost New Zealand parcel-boundary pilot
Upload `nz_colab_bundle.zip` when prompted. It includes the downloaded, prepared sample and scripts.

**This trains a small RGB-only boundary model, not the production MultiTaskNet.** Do not set `GEOCADASTRA_MODEL_WEIGHTS` to its checkpoint. No height rasters or independent survey measurements are included. Labels are current parcel-map references over March 2025 imagery; date/alignment differences require review.

Start with **Runtime → Change runtime type → CPU** for setup; choose **T4 GPU** before training if available. No paid subscription is required. A tiny sample cannot establish national accuracy.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Select nz_colab_bundle.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/geocadastra_nz')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (root / name).resolve().is_relative_to(root.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(root)
%cd /content/geocadastra_nz
%pip -q install -r scripts/colab/requirements.txt


Save checkpoints to Drive so a disconnected Colab session can resume. Downloading files does not need a GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN = '/content/drive/MyDrive/GeoCadastra/nz_rgb_pilot'
import torch, json
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Unavailable: select a GPU runtime')
manifest = json.loads(Path('data/nz_pilot/manifest.json').read_text())
print([(t['tile'], t['split'], t['parcels']) for t in manifest['tiles']])
print('\n'.join(manifest['limitations']))


Inspect the preview before training. Parcel labels need a location/date review; bright boundaries need not follow visible fences.

In [ ]:
from IPython.display import display
from PIL import Image
display(Image.open('data/nz_pilot/overview.jpg'))


First run: one epoch to measure speed and memory. A checkpoint is saved after each epoch. If memory is exhausted, use a **new run folder** with batch size 4 or 2.

In [ ]:
import subprocess, sys
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime before this training cell; CPU is supported via the script for local checks.')
base = [sys.executable, 'scripts/colab/train_nz.py', '--data', 'data/nz_pilot', '--out', RUN]
resume = ['--resume'] if Path(RUN, 'last.pt').exists() else []
subprocess.run(base + ['--epochs','1','--max-minutes','10'] + resume, check=True)


Continue up to 20 total epochs, stopping early after five epochs without validation-loss improvement. The 45-minute limit is checked **between epochs**, not a hard runtime limit. Re-run to resume after disconnects; keep the same configuration.

In [ ]:
subprocess.run(base + ['--epochs','20','--max-minutes','45','--resume'], check=True)
history=json.loads(Path(RUN,'history.json').read_text())
print('Latest:',history[-1])
print('Measured seconds per epoch:',history[-1]['seconds'])


Evaluate the untouched test location only after model choices are final. Metrics describe overlap with a three-pixel-wide parcel-map boundary band at 30 cm/pixel. They are not survey accuracy or calibrated confidence.

In [ ]:
subprocess.run(base + ['--evaluate'], check=True)
print(Path(RUN,'test_metrics.json').read_text())
display(Image.open(Path(RUN,'test_preview.png')))


Optional: rebuild the data from public sources. This downloads native 7.5 cm GeoTIFFs and produces 30 cm training copies. A refreshed parcel snapshot changes the manifest, so use a **new run directory** afterwards.

In [ ]:
# Run only if you want a fresh download; bundled data is already prepared.
# subprocess.run([sys.executable, 'scripts/colab/download_nz.py', '--out', 'data/nz_pilot'], check=True)
